# P4 Notebook-First · A2-09-NCS-MAP

## Accept Agent 4 lexical mapping handoff

| 항목 | 명세 |
|---|---|
| 목적 | Validate and load lexical top-5 candidates, matches, and the preserved unmapped row; dense reranking remains disabled. |
| 담당 Agent | `P4-A2-PIPELINE` |
| Stage ID | `A2-09-NCS-MAP` |
| 입력 | `Agent2 duty handoff`<br>`Agent4 candidates and matches` |
| 처리 | `p4.notebooks.observed_stages.run_map_posting_to_ncs_stage` 호출 |
| 출력 | `observed.posting_ncs_candidates`<br>`observed.posting_ncs_matches`<br>`acceptance handoff`<br>`four termination artifacts` |
| 선행 Gate | `REQUIREMENT_READY_AND_NCS_MAPPING_DEV_READY` |
| 후속 활용 | preprocessed track export enrichment |

In [1]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A2-PIPELINE"
STAGE_ID = 'A2-09-NCS-MAP'
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = 'ncs-agent4-acceptance-v1'
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = 'shared/handoffs/AGENT4_TO_AGENT2_NCS_MAPPING_OBSERVED_DEV.json'
OUTPUT_ROOT = "pipeline/data/exports/observed-dev/OBSERVED_DEV_20260806_01"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
PROJECT_ROOT = None
RELEASE_ROOT = None

# Injected into the executed copy by crawl.control.notebook_bundle
RUN_MODE = 'observed-dev'
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pipeline/pyproject.toml").exists() and (candidate / "pipeline/src/p4").exists():
            return candidate
    raise RuntimeError("project root not found")

PROJECT_ROOT = Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else find_project_root(Path.cwd().resolve())
PIPELINE_ROOT = PROJECT_ROOT / "pipeline"
sys.path.insert(0, str(PIPELINE_ROOT / "src"))
RELEASE_ROOT = Path(RELEASE_ROOT).resolve() if RELEASE_ROOT else PROJECT_ROOT / "crawl/observed_inputs/OBSERVED_INPUT_20260806_01"
CRAWL_ROOT = Path(os.environ.get("P4_CRAWL_ROOT", PROJECT_ROOT / "crawl")).resolve()
OUTPUT_ROOT = Path(OUTPUT_ROOT)
OUTPUT_ROOT = OUTPUT_ROOT.resolve() if OUTPUT_ROOT.is_absolute() else (PROJECT_ROOT / OUTPUT_ROOT).resolve()
CONTROL_ROOT = Path(os.environ.get("P4_CONTROL_ROOT", PROJECT_ROOT / "crawl/control")).resolve()
NCS_PROJECT_ROOT = Path(os.environ.get("P4_NCS_PROJECT_ROOT", PROJECT_ROOT)).resolve()
NCS_HANDOFF_PATH = Path(os.environ.get("P4_NCS_HANDOFF_PATH", NCS_PROJECT_ROOT / "shared/handoffs/AGENT4_TO_AGENT2_NCS_MAPPING_OBSERVED_DEV.json")).resolve()
RUN_ROOT = Path(os.environ.get("P4_NOTEBOOK_RUN_ROOT", PIPELINE_ROOT / "runs/notebooks/observed-dev/AGENT2_20260806_01")).resolve()
BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True).strip() or "DETACHED_HEAD"
GIT_HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
METADATA = {
    "agentId": AGENT_ID, "branch": BRANCH, "gitHead": GIT_HEAD,
    "contractVersion": CONTRACT_VERSION, "crawlReleaseId": CRAWL_RELEASE_ID,
    "dataVersion": DATA_VERSION, "runMode": RUN_MODE, "dataProvenance": DATA_PROVENANCE,
    "asOfDate": AS_OF_DATE, "randomSeed": RANDOM_SEED,
    "startedAt": datetime.now(timezone.utc).isoformat(),
    "inputManifestPath": INPUT_MANIFEST_PATH,
    "outputRoot": "pipeline/data/exports/observed-dev/OBSERVED_DEV_20260806_01",
    "empiricalAnalysisAllowed": EMPIRICAL_ANALYSIS_ALLOWED, "promotionAllowed": PROMOTION_ALLOWED,
    "storagePolicy": {"canonical": "DUCKDB_PARQUET", "inspectionExport": "CSV_UTF8_SIG"},
    "eligibilityColumns": ["postingEligibleFlag", "rq1EligibleFlag", "rq2EligibleFlag", "ncsEligibleFlag"],
    "highDemandScorePolicy": "ALL_NULL",
    "ksaPolicy": {"decisionId": "D-023", "status": "PROVISIONAL", "mode": "OPTIONAL_ENRICHMENT", "blocksM1": False},
    "mappingPolicy": {"mappingMode": "LEXICAL_BASELINE", "codeSetStatus": "REVIEW_REQUIRED", "goldValidatedFlag": False, "denseScore": None},
}
assert RUN_MODE == "observed-dev"
assert AGENT_ID == "P4-A2-PIPELINE" and STAGE_ID.startswith("A2-")
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == "OBSERVED_DEVELOPMENT_ONLY"
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
if (CONTROL_ROOT / "NOTEBOOK_EXECUTION_CONTRACT.schema.json").is_file():
    import jsonschema
    schema = json.loads((CONTROL_ROOT / "NOTEBOOK_EXECUTION_CONTRACT.schema.json").read_text(encoding="utf-8"))
    jsonschema.Draft202012Validator(schema, format_checker=jsonschema.FormatChecker()).validate(METADATA)
print(json.dumps(METADATA, ensure_ascii=False, indent=2))

{
  "agentId": "P4-A2-PIPELINE",
  "branch": "DETACHED_HEAD",
  "gitHead": "71edc867a8e869b78340e4fe253ced9312894077",
  "contractVersion": "2.1.2",
  "crawlReleaseId": "CRAWL_20260806_03",
  "dataVersion": "observed-dev-20260806.1",
  "runMode": "observed-dev",
  "dataProvenance": "OBSERVED_DEVELOPMENT_ONLY",
  "asOfDate": "2026-08-06",
  "randomSeed": 20260806,
  "startedAt": "2026-08-06T09:16:36.124034+00:00",
  "inputManifestPath": "shared/handoffs/AGENT4_TO_AGENT2_NCS_MAPPING_OBSERVED_DEV.json",
  "outputRoot": "pipeline/data/exports/observed-dev/OBSERVED_DEV_20260806_01",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false,
  "storagePolicy": {
    "canonical": "DUCKDB_PARQUET",
    "inspectionExport": "CSV_UTF8_SIG"
  },
  "eligibilityColumns": [
    "postingEligibleFlag",
    "rq1EligibleFlag",
    "rq2EligibleFlag",
    "ncsEligibleFlag"
  ],
  "highDemandScorePolicy": "ALL_NULL",
  "ksaPolicy": {
    "decisionId": "D-023",
    "status": "PROVISIONAL",
    "mode":

## Stage contract

**Inputs**

- `Agent2 duty handoff`
- `Agent4 candidates and matches`

**Outputs**

- `observed.posting_ncs_candidates`
- `observed.posting_ncs_matches`
- `acceptance handoff`
- `four termination artifacts`

In [3]:
from p4.notebooks.observed_stages import audit_observed_stage_inputs
STAGE = '09MapPostingToNcs'
INPUT_AUDIT = audit_observed_stage_inputs(
    STAGE,
    project_root=PROJECT_ROOT,
    release_root=RELEASE_ROOT,
    ncs_handoff_path=NCS_HANDOFF_PATH,
)
assert INPUT_AUDIT["missingInputCount"] == 0
print(json.dumps(INPUT_AUDIT, ensure_ascii=False, indent=2))

{
  "stage": "09MapPostingToNcs",
  "requiredInputCount": 3,
  "missingInputCount": 0,
  "inputNames": [
    "HANDOFF.json",
    "posting_manifest.parquet",
    "AGENT4_TO_AGENT2_NCS_MAPPING_OBSERVED_DEV.json"
  ],
  "runMode": "observed-dev",
  "dataProvenance": "OBSERVED_DEVELOPMENT_ONLY",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false
}


## Execute versioned stage module

In [4]:
from p4.notebooks.observed_stages import run_map_posting_to_ncs_stage
RESULT = run_map_posting_to_ncs_stage(
    project_root=PROJECT_ROOT,
    release_root=RELEASE_ROOT,
    crawl_root=CRAWL_ROOT,
    output_root=OUTPUT_ROOT,
    control_root=CONTROL_ROOT,
    ncs_handoff_path=NCS_HANDOFF_PATH,
    ncs_project_root=NCS_PROJECT_ROOT,
    run_root=RUN_ROOT,
)
if FAIL_ON_GATE:
    assert RESULT["qualityStatus"] == "PASS", RESULT
print(json.dumps(RESULT, ensure_ascii=False, indent=2))

{
  "stageManifest": "runs/notebooks/observed-dev/MASTER_20260806_01/artifacts/09MapPostingToNcs/stage_manifest.json",
  "stageMetrics": "runs/notebooks/observed-dev/MASTER_20260806_01/artifacts/09MapPostingToNcs/stage_metrics.json",
  "stageQuality": "runs/notebooks/observed-dev/MASTER_20260806_01/artifacts/09MapPostingToNcs/stage_quality.csv",
  "checksums": "runs/notebooks/observed-dev/MASTER_20260806_01/artifacts/09MapPostingToNcs/CHECKSUMS.sha256",
  "qualityStatus": "PASS"
}


## Stage summary

In [5]:
from IPython.display import display
import pandas as pd

SUMMARY = pd.DataFrame([
    {"field": "stage", "value": STAGE},
    {"field": "qualityStatus", "value": RESULT["qualityStatus"]},
    {"field": "stageManifest", "value": RESULT["stageManifest"]},
    {"field": "stageMetrics", "value": RESULT["stageMetrics"]},
    {"field": "stageQuality", "value": RESULT["stageQuality"]},
    {"field": "checksums", "value": RESULT["checksums"]},
])
display(SUMMARY)

,field,value
0,stage,09MapPostingToNcs
1,qualityStatus,PASS
2,stageManifest,runs/notebooks/observed-dev/MASTER_20260806_01...
3,stageMetrics,runs/notebooks/observed-dev/MASTER_20260806_01...
4,stageQuality,runs/notebooks/observed-dev/MASTER_20260806_01...
5,checksums,runs/notebooks/observed-dev/MASTER_20260806_01...


## Termination contract

In [6]:
REQUIRED_TERMINATION_ARTIFACTS = (
    "stage_manifest.json", "stage_metrics.json", "stage_quality.csv", "CHECKSUMS.sha256",
)
artifact_root = RUN_ROOT / "artifacts" / STAGE
missing = [name for name in REQUIRED_TERMINATION_ARTIFACTS if not (artifact_root / name).is_file()]
assert not missing, missing
assert RESULT["qualityStatus"] == "PASS"
print(json.dumps({
    "stage": STAGE,
    "terminationArtifacts": list(REQUIRED_TERMINATION_ARTIFACTS),
    "artifactRoot": str(artifact_root.relative_to(PROJECT_ROOT)),
    "empiricalAnalysisAllowed": False,
    "promotionAllowed": False,
}, ensure_ascii=False, indent=2))

{
  "stage": "09MapPostingToNcs",
  "terminationArtifacts": [
    "stage_manifest.json",
    "stage_metrics.json",
    "stage_quality.csv",
    "CHECKSUMS.sha256"
  ],
  "artifactRoot": "pipeline/runs/notebooks/observed-dev/MASTER_20260806_01/artifacts/09MapPostingToNcs",
  "empiricalAnalysisAllowed": false,
  "promotionAllowed": false
}
